In [1]:
from pathlib import Path
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from dotenv import load_dotenv


env_path = Path.cwd() / ".env"
if not env_path.exists():
    env_path = Path.cwd().parent / ".env"
load_dotenv(env_path)  # Load environment variables from workspace root .env file


llm = ChatGroq(
    model="qwen/qwen3-32b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2,
)

c:\projects\learn-rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
question = HumanMessage('tell me about the earth in 3 points')
system = SystemMessage('You are elemetary teacher. You answer in short sentences.')


messages = [system, question]
response = llm.invoke(messages)


print(response.content)


1. Earth is the third planet from the Sun in our solar system.  
2. It is the only planet known to support life, with water, air, and diverse ecosystems.  
3. Earth has layers: a solid crust, a hot mantle, and a dense core.


In [ ]:
# Structured Prompting: Organized hierarchical structure
structured_prompt = """
TASK: Write a blog post outline

CONTEXT:
- Target audience: Beginners
- Topic: Cloud Computing
- Length: 5 sections

REQUIREMENTS:
1. Include introduction and conclusion
2. Each section should have 2-3 key points
3. Use simple, non-technical language

OUTPUT FORMAT:
I. Introduction
   A. Point 1
   B. Point 2
II. [Next Section]
...
"""

question_structured = HumanMessage(structured_prompt)
response_structured = llm.invoke([question_structured])
print("Structured Prompting Response:")
print(response_structured.content)

## 10. Structured Prompting
Use a structured format (like templates) with clear sections and hierarchies.

In [3]:
# Output Formatting: Request specific output structure
format_prompt = """
List 3 fruits and their nutritional benefits.

Format your response as JSON:
{
  "fruits": [
    {"name": "Apple", "benefits": ["Fiber", "Vitamin C"]},
    ...
  ]
}
"""

question_format = HumanMessage(format_prompt)
response_format = llm.invoke([question_format])
print("Output Formatting Response:")
print(response_format.content)
print("\n")

Output Formatting Response:
{  "fruits": [
    {"name": "Apple", "benefits": ["Fiber", "Vitamin C"]},
    {"name": "Banana", "benefits": ["Potassium", "Vitamin B6"]},
    {"name": "Orange", "benefits": ["Vitamin C", "Folate"]}
  ]
}




## 9. Output Formatting
Specify exactly how you want the response formatted (JSON, bullet points, table, etc.).

In [4]:
# Retrieval-Augmented: Include retrieved context in prompt
retrieved_context = """
Retrieved Document: Machine Learning Basics
- ML is a subset of AI that enables systems to learn from data
- Supervised learning requires labeled training data
- Unsupervised learning finds patterns in unlabeled data
"""

rag_prompt = f"""
{retrieved_context}

Based on the document above, what is the difference between supervised and unsupervised learning?
"""

question_rag = HumanMessage(rag_prompt)
response_rag = llm.invoke([question_rag])
print("Retrieval-Augmented Response:")
print(response_rag.content)
print("\n")

Retrieval-Augmented Response:
The difference between **supervised** and **unsupervised learning** lies in the type of data they use and their objectives:  

1. **Supervised Learning**:  
   - **Requires labeled training data** (input-output pairs).  
   - The model learns to map inputs to known outputs (e.g., predicting house prices based on features like size or location).  
   - Common tasks: Classification (e.g., spam detection) and regression (e.g., predicting numerical values).  

2. **Unsupervised Learning**:  
   - **Uses unlabeled data** (no predefined outputs).  
   - The model identifies hidden patterns or structures in the data (e.g., grouping customers into segments based on purchasing behavior).  
   - Common tasks: Clustering (e.g., customer segmentation) and dimensionality reduction (e.g., simplifying data for visualization).  

**Key Contrast**: Supervised learning relies on explicit feedback (labels) to train models, while unsupervised learning explores data autonomous

## 8. Retrieval-Augmented Prompting
Include relevant context or retrieved documents in the prompt to ground responses.

In [5]:
# Instruction Following: Clear structured instructions
instruction_prompt = """
Write a product description following these rules:
1. Maximum 100 words
2. Include 3 key benefits
3. Use active voice
4. Include a call-to-action
5. Do not use marketing clichés

Product: Wireless Headphones
"""

question_instruction = HumanMessage(instruction_prompt)
response_instruction = llm.invoke([question_instruction])
print("Instruction Following Response:")
print(response_instruction.content)
print("\n")

Instruction Following Response:
Experience clear, immersive sound with these wireless headphones. Their ergonomic design ensures comfort during extended use, and a single charge provides up to 20 hours of playback. Connect effortlessly via Bluetooth for reliable device pairing. Ideal for music, calls, or podcasts. Buy now to upgrade your listening experience.  

(98 words)




## 7. Instruction Following
Provide clear, structured instructions with specific constraints and rules.

In [6]:
# Role-Based Prompting: Assign a specific role to the model
system_role = SystemMessage("""
You are an experienced software architect with 15 years of experience.
Always provide practical, production-ready solutions.
Consider scalability, security, and performance.
""")

question_role = HumanMessage("How should I design a user authentication system?")
response_role = llm.invoke([system_role, question_role])
print("Role-Based Response:")
print(response_role.content)
print("\n")

Role-Based Response:
Designing a secure, scalable, and production-ready user authentication system requires a layered approach. Below is a structured design with practical implementation details:

---

### **Core Components**
1. **User Registration**
   - **Email/Username + Password**: 
     - Validate email format and enforce unique usernames/emails.
     - Use **bcrypt** or **Argon2** for password hashing (never MD5/SHA1).
     - Add **salt** automatically via the hashing library.
   - **Email Verification**:
     - Send a time-limited token (e.g., 1-hour expiry) via email.
     - Store the token in a `verification_tokens` table with expiration.

2. **Login System**
   - **Password Verification**:
     - Compare the provided password with the stored hash using `bcrypt.compare()` or equivalent.
   - **Rate Limiting**:
     - Throttle login attempts (e.g., 5 attempts/minute per IP/user) using Redis or a similar in-memory store.
   - **Session Management**:
     - Use **secure, HTTP-onl

## 6. System Prompts & Role-Based Prompting
Set a specific role/context for the model to follow throughout the conversation.

In [ ]:
# Prompt Templating: Reusable template structure
def create_prompt_template(context, question, format_instruction):
    return f"""
Context: {context}
Question: {question}
Format: {format_instruction}
"""

template = create_prompt_template(
    context="Python is a programming language",
    question="What are the main features of Python?",
    format_instruction="Provide 3 key features as bullet points"
)

question_template = HumanMessage(template)
response_template = llm.invoke([question_template])
print("Prompt Template Response:")
print(response_template.content)
print("\n")

## 5. Prompt Templating
Use reusable template structures for consistent prompting patterns.

In [ ]:
# Tree-of-Thought: Explore multiple solutions
tot_prompt = """
Problem: How can I improve my productivity?

Explore 3 different approaches:
1. Time management approach
2. Technology approach  
3. Habit-building approach

For each approach, list pros and cons, then recommend the best one.
"""

question_tot = HumanMessage(tot_prompt)
response_tot = llm.invoke([question_tot])
print("Tree-of-Thought Response:")
print(response_tot.content)
print("\n")

## 4. Tree-of-Thought
Explore multiple reasoning paths and choose the best one.

In [7]:
# Chain-of-Thought: Ask for step-by-step reasoning
cot_prompt = "If a car travels 60 km/h for 2 hours, how far does it go? Think step by step."

question_cot = HumanMessage(cot_prompt)
response_cot = llm.invoke([SystemMessage("Explain your reasoning step by step."), question_cot])
print("Chain-of-Thought Response:")
print(response_cot.content)
print("\n")

Chain-of-Thought Response:
To determine how far the car travels, we use the fundamental relationship between distance, speed, and time:

$$
\text{Distance} = \text{Speed} \times \text{Time}
$$

**Step 1: Identify the given values**  
- **Speed**: 60 km/h  
- **Time**: 2 hours  

**Step 2: Apply the formula**  
$$
\text{Distance} = 60 \, \text{km/h} \times 2 \, \text{h}
$$

**Step 3: Perform the calculation**  
$$
\text{Distance} = 120 \, \text{km}
$$

**Step 4: Verify units**  
The "hours" unit cancels out, leaving the result in kilometers, which is correct for distance.

**Conclusion**: The car travels **120 kilometers**.




## 3. Chain-of-Thought (CoT)
Ask the model to explain its reasoning step-by-step before giving the final answer.

In [ ]:
# Few-Shot: Provide examples to guide the model
few_shot_prompt = """
You are a sentiment analyzer. Classify the sentiment as Positive, Negative, or Neutral.

Examples:
- "I love this product!" → Positive
- "This is terrible." → Negative
- "It is a chair." → Neutral

Now classify: "The service was amazing!"
"""

question_few_shot = HumanMessage(few_shot_prompt)
response_few_shot = llm.invoke([question_few_shot])
print("Few-Shot Response:")
print(response_few_shot.content)
print("\n")

## 2. Few-Shot Prompting
Provide examples before asking the model to perform the task. This improves accuracy.

In [ ]:
# Zero-Shot: Ask directly without examples
question_zero_shot = HumanMessage("What is photosynthesis?")
response_zero_shot = llm.invoke([SystemMessage("You are a biology teacher."), question_zero_shot])
print("Zero-Shot Response:")
print(response_zero_shot.content)
print("\n")

# Prompting Strategies for LLMs

## 1. Zero-Shot Prompting
Direct instruction without examples. The model responds based on training knowledge.